# 07a — Turnout trends 2011–2021

**Goal:** describe how municipal turnout moved across the last three local elections, and where it fell most.

**Reads:** `INTERIM / "results_lge_2011_2021.csv"` (made by `01_results_lge`)
**Writes:** charts to `FIGURES / "turnout_*.png"` (for slides)

Run the two setup cells below first, every time.

## Setup

In [ ]:
import sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/Thato20-M/democratic-funnel.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/democratic-funnel")
    url = REPO_URL
    try:                                    # private repo: token from Colab Secrets
        from google.colab import userdata
        url = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
    except Exception:
        print("No GITHUB_TOKEN in Colab Secrets - cloning will fail for the private repo")
    if REPO_DIR.exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", url, str(REPO_DIR)], check=True)
else:
    REPO_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "src" / "config.py").exists())

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print("Repo:", REPO_DIR)

In [ ]:
from src import config
from src.config import RAW, INTERIM, PROCESSED, MODELS, FIGURES, SEED
from src.io_utils import read_any, clean_columns, inventory
from src.privacy import drop_personal

import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import warnings; warnings.filterwarnings("ignore", category=FutureWarning)

config.make_dirs(); config.set_seed()
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid"); plt.rcParams["figure.dpi"] = 110
print(config.describe())

## Chart style

Two series colours only (blue, orange: checked colour-blind safe), grey for context,
hairline grid. Every chart is saved to `FIGURES` as a PNG for the slides.

In [ ]:
from matplotlib.ticker import PercentFormatter

BLUE, ORANGE = "#2a78d6", "#eb6834"                 # series colours
INK, INK2, MUTED = "#0b0b0b", "#52514e", "#898781"  # text: primary, secondary, muted
GRID, AXIS, SURFACE = "#e1e0d9", "#c3c2b7", "#fcfcfb"

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": AXIS, "axes.labelcolor": INK2, "axes.titlecolor": INK,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8, "grid.linestyle": "-",
    "xtick.color": MUTED, "ytick.color": MUTED, "xtick.labelcolor": INK2, "ytick.labelcolor": INK2,
    "legend.frameon": False, "legend.labelcolor": INK2, "font.size": 10,
})

def tidy(ax, grid="y"):
    """Hairline grid on one axis only, no box."""
    for side in ("top", "right", "left"):
        ax.spines[side].set_visible(False)
    ax.grid(axis="x" if grid == "y" else "y", visible=False)
    ax.tick_params(length=0)

def save(fig, name):
    fig.savefig(FIGURES / f"{name}.png", dpi=200, bbox_inches="tight")
    plt.show()

## Load

In [ ]:
r = pd.read_csv(INTERIM / "results_lge_2011_2021.csv")

# Some IEC names are in capitals (e.g. "KAREEBERG"): title-case those only
letters = r["muni_name"].str.replace(r"[^A-Za-z]", "", regex=True)
r["muni_name"] = r["muni_name"].where(~letters.str.isupper(), r["muni_name"].str.title())

# One row per municipality (2021 names/boundaries), turnout per election in columns
wide = (r[r["election"] == 2021].set_index("muni_code")[["province", "muni_name", "muni_type", "registered"]]
        .join(r.pivot(index="muni_code", columns="election", values="turnout")))
wide["change_16_21"] = (wide[2021] - wide[2016]) * 100          # percentage points

comparable = r[r["election"] == 2011].set_index("muni_code")["comparable_2011"].eq(1)
both = wide.index.intersection(comparable[comparable].index)
wide["change_11_16"] = np.nan
wide.loc[both, "change_11_16"] = (wide.loc[both, 2016] - wide.loc[both, 2011]) * 100

print(r.shape, "->", wide.shape)

## 1. National turnout

Stable from 2011 to 2016, then a 12-point fall in 2021 (the COVID election, `covid_shock` = 1).

In [ ]:
national = r.groupby("election")[["votes_cast", "registered"]].sum()
national["turnout"] = national["votes_cast"] / national["registered"]

fig, ax = plt.subplots(figsize=(6.5, 3.6))
ax.plot(national.index, national["turnout"], color=BLUE, lw=2, marker="o", ms=8,
        markeredgecolor=SURFACE, markeredgewidth=2)
for year, t in national["turnout"].items():
    ax.annotate(f"{t:.1%}", (year, t), xytext=(0, 10), textcoords="offset points",
                ha="center", color=INK, fontsize=10, fontweight="bold")
ax.set(title="National turnout, local elections", xticks=national.index, ylim=(0, 0.7), xlim=(2009.5, 2022.5))
ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
tidy(ax)
save(fig, "turnout_national")
national.assign(turnout=lambda d: (d["turnout"] * 100).round(1))

## 2. Metros fell hardest and dropped below the smaller municipalities

In 2016 the 8 metros voted **more** than the local municipalities; in 2021 they voted **less**.

In [ ]:
by_type = r.groupby(["election", "muni_type"])[["votes_cast", "registered"]].sum()
by_type = (by_type["votes_cast"] / by_type["registered"]).unstack()

fig, ax = plt.subplots(figsize=(6.5, 3.6))
for col, colour, label in [("metro", BLUE, "Metros (8)"), ("local", ORANGE, "Local municipalities")]:
    ax.plot(by_type.index, by_type[col], color=colour, lw=2, marker="o", ms=8,
            markeredgecolor=SURFACE, markeredgewidth=2, label=label)
    end = by_type[col].iloc[-1]
    ax.annotate(f"{label.split(' (')[0]} {end:.1%}", (2021, end), xytext=(10, 0), textcoords="offset points",
                va="center", color=INK2, fontsize=9)
ax.set(title="Turnout: metros vs local municipalities", xticks=by_type.index, ylim=(0.35, 0.65), xlim=(2009.5, 2025))
ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
ax.legend(loc="lower left")
tidy(ax)
save(fig, "turnout_metro_vs_local")
(by_type * 100).round(1)

## 3. Every municipality, every election

Each dot is a municipality; the black bar is the median. The whole distribution shifted down in 2021.

In [ ]:
rng = np.random.default_rng(SEED)
fig, ax = plt.subplots(figsize=(6.5, 4))
labels = []
for i, (year, g) in enumerate(r.groupby("election")):
    x = i + rng.uniform(-0.2, 0.2, len(g))
    ax.scatter(x, g["turnout"], s=14, color=BLUE, alpha=0.45, linewidths=0)
    med = g["turnout"].median()
    ax.plot([i - 0.3, i + 0.3], [med, med], color=INK, lw=2)
    labels.append(f"{year}\nmedian {med:.1%}")
ax.set(title="Municipal turnout by election (bar = median)", xticks=range(3), xticklabels=labels, xlim=(-0.5, 2.5))
ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
tidy(ax)
save(fig, "turnout_distribution")
r.groupby("election")["turnout"].describe().mul(100).round(1).drop(columns="count")

## 4. How much each municipality fell, 2016 → 2021

2016 and 2021 have identical boundaries, so all 213 municipalities can be compared.

In [ ]:
chg = wide["change_16_21"]
fig, ax = plt.subplots(figsize=(6.5, 3.6))
ax.hist(chg, bins=np.arange(-25, 4, 1), color=BLUE, edgecolor=SURFACE, linewidth=1)
ax.axvline(0, color=INK2, lw=1)
ax.axvline(chg.median(), color=INK, lw=2)
top = ax.get_ylim()[1] * 1.15
ax.set_ylim(0, top)
ax.annotate(f"median {chg.median():.1f} pts", (chg.median(), top * 0.95),
            xytext=(-6, 0), textcoords="offset points", ha="right", va="center",
            color=INK, fontsize=9, fontweight="bold")
ax.annotate("no change", (0, top * 0.95), xytext=(6, 0), textcoords="offset points",
            va="center", color=INK2, fontsize=9)
ax.annotate(f"{(chg < 0).sum()} of {len(chg)} fell", (-24.5, top * 0.8), color=INK2, fontsize=9)
ax.set(title="Change in turnout 2016 → 2021, per municipality", xlabel="percentage points", ylabel="municipalities")
tidy(ax)
save(fig, "turnout_change_16_21")

print(f"2016 -> 2021: fell in {(chg < 0).sum()} of {chg.notna().sum()}, median {chg.median():.1f} pts, "
      f"range {chg.min():.1f} to {chg.max():.1f}")
c11 = wide["change_11_16"].dropna()
print(f"2011 -> 2016 (same boundaries, {len(c11)} munis): fell in {(c11 < 0).sum()}, median {c11.median():.1f} pts")

## 5. Provinces

Western Cape and Gauteng fell most; Limpopo and the Northern Cape least. The Northern Cape is now the highest-turnout province.

In [ ]:
prov = r.groupby(["province", "election"])[["votes_cast", "registered"]].sum()
prov = (prov["votes_cast"] / prov["registered"]).unstack()
prov["change"] = (prov[2021] - prov[2016]) * 100
prov = prov.sort_values("change", ascending=False)       # biggest fall at the top of the chart

fig, ax = plt.subplots(figsize=(6.5, 4.2))
y = np.arange(len(prov))
ax.hlines(y, prov[2021], prov[2016], color=AXIS, lw=2)
ax.scatter(prov[2016], y, s=64, color=MUTED, zorder=3, label="2016", edgecolors=SURFACE, linewidths=2)
ax.scatter(prov[2021], y, s=64, color=BLUE, zorder=3, label="2021", edgecolors=SURFACE, linewidths=2)
for yi, (_, row) in zip(y, prov.iterrows()):
    ax.annotate(f"{row['change']:.1f}", (row[2016], yi), xytext=(10, 0), textcoords="offset points",
                va="center", color=INK2, fontsize=9)
ax.set(title="Turnout by province, 2016 → 2021 (change in points)", yticks=y, yticklabels=prov.index, xlim=(0.38, 0.68))
ax.xaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.08), ncols=2)
tidy(ax, grid="x")
save(fig, "turnout_provinces")
(prov[[2011, 2016, 2021]] * 100).round(1).assign(change_16_21=prov["change"].round(1))

## 6. Past turnout predicts future turnout

Each dot is a municipality. Dots below the grey line fell. Places that voted more in 2016 still voted more in 2021,
so last election's turnout is a strong baseline for forecasting 2026.

In [ ]:
metro = wide["muni_type"] == "metro"
fig, ax = plt.subplots(figsize=(6, 5.4))
ax.plot([0.3, 0.8], [0.3, 0.8], color=AXIS, lw=1)
ax.annotate("line = no change\nbelow the line = turnout fell", (0.79, 0.31), ha="right", va="bottom",
            color=INK2, fontsize=9)
ax.scatter(wide.loc[~metro, 2016], wide.loc[~metro, 2021], s=22, color=BLUE, alpha=0.6,
           edgecolors=SURFACE, linewidths=1, label="Local municipality")
ax.scatter(wide.loc[metro, 2016], wide.loc[metro, 2021], s=64, color=ORANGE,
           edgecolors=SURFACE, linewidths=2, label="Metro")
nudge = {"MAN": (-8, 6), "EKU": (8, -1), "JHB": (-8, -4), "BUF": (-8, 4)}   # keep close metros apart
for code_, row in wide[metro].iterrows():
    dx, dy = nudge.get(code_, (7, -3))
    ax.annotate(code_, (row[2016], row[2021]), xytext=(dx, dy), textcoords="offset points",
                ha="right" if dx < 0 else "left", va="center", color=INK, fontsize=8,
                bbox=dict(boxstyle="round,pad=0.15", fc=SURFACE, ec="none", alpha=0.85))
ax.set(title="Turnout 2016 vs 2021", xlabel="2016", ylabel="2021", xlim=(0.4, 0.8), ylim=(0.3, 0.75))
ax.xaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
ax.legend(loc="upper left")
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
ax.tick_params(length=0)
save(fig, "turnout_2016_vs_2021")

print("correlation 2016 vs 2021 turnout:", round(wide[2016].corr(wide[2021]), 2))
print("correlation 2011 vs 2016 turnout (same boundaries):",
      round(wide.loc[both, 2011].corr(wide.loc[both, 2016]), 2))
print("correlation 2016 turnout vs size of the 2021 fall:", round(wide[2016].corr(wide["change_16_21"]), 2))

## 7. Bigger municipalities fell more

Municipalities split into four equal groups by number of registered voters (2021).

In [ ]:
wide["size_group"] = pd.qcut(wide["registered"], 4,
                             labels=["Smallest 25%", "Second 25%", "Third 25%", "Largest 25%"])
size = wide.groupby("size_group", observed=True)["change_16_21"].median()

fig, ax = plt.subplots(figsize=(6.5, 3))
y = np.arange(len(size))[::-1]
ax.barh(y, size.values, color=BLUE, height=0.6)
for yi, v in zip(y, size.values):
    ax.annotate(f"{v:.1f}", (v, yi), xytext=(-6, 0), textcoords="offset points",
                ha="right", va="center", color=INK2, fontsize=9)
ax.axvline(0, color=INK2, lw=1)
ax.set(title="Median change in turnout 2016 → 2021, by municipality size", yticks=y,
       yticklabels=size.index, xlabel="percentage points", xlim=(-13, 0.5))
tidy(ax, grid="x")
save(fig, "turnout_by_size")
size.round(1).to_frame("median change (pts)")

## 8. Tables: biggest falls, rises, lowest turnout

In [ ]:
table = wide[["province", "muni_name", "muni_type", 2016, 2021, "change_16_21"]].copy()
table[[2016, 2021]] = (table[[2016, 2021]] * 100).round(1)
table["change_16_21"] = table["change_16_21"].round(1)

print("Biggest falls 2016 -> 2021")
display(table.sort_values("change_16_21").head(10))
print("Rose or barely changed")
display(table.sort_values("change_16_21").tail(6))
print("Lowest turnout 2021")
display(table.sort_values(2021).head(10))

## 9. Registration also shrank

In [ ]:
reg = r.pivot(index="muni_code", columns="election", values="registered")
growth = (reg[2021] / reg[2016] - 1) * 100
print(f"Registered voters 2016 -> 2021: {(reg[2021].sum() / reg[2016].sum() - 1) * 100:.2f}% nationally")
print(f"Register shrank in {(growth < 0).sum()} of {growth.notna().sum()} municipalities")

spoilt = r.groupby("election")[["spoilt", "votes_cast"]].sum()
print("\nSpoilt votes as % of votes cast:")
print((spoilt["spoilt"] / spoilt["votes_cast"] * 100).round(2).to_string())

## Key findings

Numbers from the September 2026 version of `results_lge_2011_2021.csv`; the cells above recompute them.

1. **Stable, then a collapse.** National turnout was 57.4% (2011) and 57.7% (2016), then fell to 45.6% in 2021.
   It fell in 209 of 213 municipalities (median −8.8 points).
2. **The cities drove the national fall.** Metros went from 59.5% to 43.7% and are now *below* the local
   municipalities (46.9%). The largest quarter of municipalities fell a median −11.1 points, the smallest −7.5.
3. **Provinces reshuffled.** Western Cape (−14.9) and Gauteng (−14.3) fell most; Limpopo (−5.9) least.
   The Northern Cape (53.3%) is now the highest-turnout province.
4. **Turnout is persistent.** Correlation 0.78 between 2016 and 2021 turnout (0.88 between 2011 and 2016):
   a strong baseline for the 2026 forecast. The size of the 2021 fall barely depends on 2016 turnout (−0.12).
5. **Registration shrank too:** −0.5% nationally 2016 → 2021, and in 112 of 213 municipalities, while the
   population grew. First evidence of the registration leak; stage 02 and the census will measure it.
6. **Spoilt votes are flat** at about 1.6–1.7% of votes cast; not part of the story.

**Caveats:** 2021 is the only COVID election; the 2024 national election (stage 03) shows whether the dip persisted.
The lowest-turnout places in 2021 (Rustenburg, Mafikeng, Emalahleni, Govan Mbeki) include mining areas; whether
migrant workers registered elsewhere explains this is a hypothesis to test with census data.